# SCPT-RL — End-to-end Training, Evaluation & Reporting

This notebook runs the full **SCPT-RL** pipeline (Sparse Coupled Placement
Transformer with Reinforcement Learning) on a **GPU-backed** runtime
(Google Colab or Kaggle). It is a 1:1 wrapper around the existing
command-line entrypoints — `scripts/train.py`, `scripts/eval.py`, and
`scripts/visualize.py` — so the same code that runs in CI also runs here.

## Pipeline overview

1. **Install** Python dependencies + build the Rust `pcb_parser` /
   `pcb_router` PyO3 wheels via `maturin`.
2. **Clone / copy** the repository (or pull the pre-installed one on Kaggle).
3. **Train** — BC warm-start + PPO-EAL fine-tuning (writes
   `runs/<run>/checkpoints/iter_*.pt` + `log.jsonl`).
4. **Evaluate** the final checkpoint on held-out boards.
5. **Visualize** training curves, Lagrangian multipliers, constraint
   violations, and the policy's placement heatmap.
6. **Report** — assemble a Markdown summary, downloadable as a single cell.

> **Runtime tips**
>
> - On Colab: `Runtime → Change runtime type → T4 / A100 GPU`.
> - On Kaggle: enable the **GPU** accelerator in the right-hand panel.
> - The free Colab T4 has ~15 GB VRAM. We default to the
   `default_low_batch.yaml` config which is sized to fit comfortably. If you
   have an A100, switch to `default.yaml` for full-scale training.

## 0. Runtime environment & GPU check

Detects whether we're in Colab or Kaggle, prints the device, and verifies
that PyTorch can see a CUDA GPU.

In [ ]:
import os, sys, subprocess, platform

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
PLATFORM = 'colab' if IS_COLAB else ('kaggle' if IS_KAGGLE else 'local')
print(f'Platform detected: {PLATFORM}')
print(f'Python: {sys.version.split()[0]}  ({platform.machine()})')

# Make sure we're on a GPU runtime. Fall back to a clear error otherwise.
import torch
if not torch.cuda.is_available():
    print('\u26a0  WARNING: CUDA is not available. Training will run on CPU and be very slow.')
    DEVICE = torch.device('cpu')
else:
    DEVICE = torch.device('cuda')
    print(f'\u2705 GPU: {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
    torch.cuda.empty_cache()
print(f'Device: {DEVICE}')

## 1. Install Python dependencies

We pin the same versions declared in `pyproject.toml`. The `maturin` step
is only needed if the Rust wheel is not already shipped; the install is a
no-op otherwise.

In [ ]:
%pip -q install --upgrade pip
%pip -q install \
    'torch>=2.0' \
    'torch-geometric>=2.4' \
    'gymnasium>=0.29' \
    'numpy>=1.24' \
    'pyyaml>=6.0' \
    'matplotlib>=3.7' \
    'tqdm>=4.65' \
    'maturin>=1.4'

## 2. Get the SCPT-RL source tree

- On Colab: clone from GitHub (replace the URL with your fork if needed).
- On Kaggle: either attach the repo as a dataset, or clone as below.
- Locally: reuses the working directory.

In [ ]:
REPO_URL = 'https://github.com/your-org/scpt-rl.git'   # <-- replace with your fork if needed
REPO_DIR = '/content/scpt-rl' if IS_COLAB else '/kaggle/working/scpt-rl' if IS_KAGGLE else os.getcwd()

if not os.path.isdir(os.path.join(REPO_DIR, '.git')) and not os.path.isdir(os.path.join(REPO_DIR, 'src', 'scpt')):
    print(f'Cloning {REPO_URL} into {REPO_DIR} ...')
    subprocess.check_call(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR])
else:
    print(f'Using existing repository at {REPO_DIR}')

%cd $REPO_DIR
print('Working dir:', os.getcwd())
print('Contents:', sorted(os.listdir('.'))[:10], '...')

## 3. Build the Rust `pcb_parser` / `pcb_router` wheels

The env and BC pipeline both call into the Rust core for high-speed DRC,
HPWL, and net-topology primitives. Building takes ~2-4 min on a T4.

In [ ]:
# Build the Rust extensions. Skipped automatically if already importable.
try:
    import pcb_parser   # type: ignore
    import pcb_router   # type: ignore
    print('Rust wheels already importable \u2014 skipping maturin build.')
except Exception:
    print('Building Rust wheels with maturin (this takes a few minutes) ...')
    subprocess.check_call(['bash', 'scripts/build_rust.sh'])
    print('Rust wheels built successfully.')

In [ ]:
# Make `scpt.*` importable and confirm the GPU sees our model tensors.
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
import scpt  # noqa: F401  (sanity check)
import torch
x = torch.randn(2, 3, device=DEVICE)
print('torch ok on', x.device, '| sum =', float(x.sum()))

## 4. Choose a training config

Pick a config that matches your budget:

| Config                          | Hidden d | Boards  | Iters | Steps/iter | Use case              |
|---------------------------------|---------:|--------:|------:|-----------:|-----------------------|
| `configs/smoke.yaml`            |       32 |       2 |     5 |         16 | Sanity test (\u2248 30 s) |
| `configs/default_low_batch.yaml`|      256 |       6 | 1 000 |        128 | Free Colab T4 / Kaggle |
| `configs/default.yaml`         |      256 |       6 | 1 000 |        128 | A100 / V100           |
| `configs/default_high_res.yaml` |      256 |       6 | 1 000 |        128 | Larger boards, less VRAM |

In [ ]:
# --- Edit this cell to change config ---
CONFIG = 'configs/default_low_batch.yaml'   # GPU-friendly default for free Colab
RUN_DIR = 'runs/colab_run'
RESUME  = ''                                # path to checkpoint to resume from, or empty
SKIP_PPO = False                           # set True to do BC warm-start only
SKIP_BC  = False                           # set True to skip BC (e.g. when resuming)
NO_WANDB = True                            # W&B is optional and disabled by default
assert os.path.exists(CONFIG), f'Config not found: {CONFIG}'
print(f'Using config: {CONFIG}\nRun dir : {RUN_DIR}')

## 5. (Optional) Sanity test with `smoke.yaml`

A 30-second CPU/GPU sanity run that proves every component (Rust bridge,
BC dataset, PPO-EAL trainer) loads correctly before you commit to a real
training job. Skip this if you want to go straight to a full run.

In [ ]:
RUN_SMOKE = False  # flip to True to run the 30-second sanity test first

if RUN_SMOKE:
    cmd = [
        sys.executable, 'scripts/train.py',
        '--config', 'configs/smoke.yaml',
        '--run-dir', 'runs/smoke',
    ]
    if NO_WANDB:
        cmd.append('--no-wandb')
    print(' '.join(cmd))
    subprocess.check_call(cmd)

## 6. Run training (BC + PPO-EAL)

Streams the script's stdout live. Expect ~30-60 s/iter on T4, ~10-20 s/iter
on A100. Checkpoints land in `runs/<run>/checkpoints/`, the JSONL log in
`runs/<run>/log.jsonl`.

In [ ]:
import time

cmd = [
    sys.executable, 'scripts/train.py',
    '--config', CONFIG,
    '--run-dir', RUN_DIR,
]
if NO_WANDB:
    cmd.append('--no-wandb')
if SKIP_BC:
    cmd.append('--ppo-only')
if SKIP_PPO:
    cmd.append('--bc-only')
if RESUME:
    cmd += ['--resume', RESUME]

print('Running:', ' '.join(cmd))
t0 = time.perf_counter()
subprocess.check_call(cmd)
print(f'\nTraining finished in {(time.perf_counter() - t0) / 60:.1f} min.')

In [ ]:
# Quick peek at the run directory.
ckpt_dir = os.path.join(RUN_DIR, 'checkpoints')
log_path = os.path.join(RUN_DIR, 'log.jsonl')
ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt')) if os.path.isdir(ckpt_dir) else []
print(f'Checkpoints ({len(ckpts)}):')
for c in ckpts[-5:]:
    print(' ', os.path.join(ckpt_dir, c))
if os.path.exists(log_path):
    n = sum(1 for _ in open(log_path))
    print(f'Log lines: {n}')
FINAL_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else None
print(f'Final checkpoint: {FINAL_CKPT}')

## 7. Evaluate the final checkpoint

Runs greedy-policy rollouts on each board in `env.board_paths`, plus a
BC eval loss measurement. Results are written to
`runs/<run>/eval.json`.

In [ ]:
if FINAL_CKPT is None:
    raise SystemExit('No checkpoint found \u2014 did training succeed?')

EVAL_OUT = os.path.join(RUN_DIR, 'eval.json')
N_EPISODES = 2   # episodes per board during eval (increase for tighter estimates)

cmd = [
    sys.executable, 'scripts/eval.py',
    '--checkpoint', FINAL_CKPT,
    '--config', CONFIG,
    '--output', EVAL_OUT,
    '--n-episodes', str(N_EPISODES),
]
# Pull board paths from the config so eval matches the training set.
import yaml
with open(CONFIG) as f:
    cfg_dict = yaml.safe_load(f)
for p in cfg_dict.get('env', {}).get('board_paths', []):
    cmd += ['--boards', p]

print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)
print(f'\nResults written to {EVAL_OUT}')

In [ ]:
import json
with open(EVAL_OUT) as f:
    eval_results = json.load(f)

print('=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
if eval_results.get('mean_reward') is not None:
    print(f'  Mean reward : {eval_results["mean_reward"]:.4f}')
if eval_results.get('bc_eval_loss') is not None:
    print(f'  BC eval loss: {eval_results["bc_eval_loss"]:.4f}')
if eval_results.get('mean_costs'):
    print('  Mean costs  :')
    for k, v in eval_results['mean_costs'].items():
        print(f'    {k:>14s}: {v:.4f}')
if eval_results.get('checkpoint_lambdas'):
    print('  Lambdas     :')
    for k, v in eval_results['checkpoint_lambdas'].items():
        print(f'    \u03bb_{k:>10s}: {v:.6f}')
print('=' * 60)

## 8. Visualize training curves

Re-uses the same plotting code that ships with the project. PNGs land in
`runs/<run>/plots/` and are displayed inline below.

In [ ]:
PLOT_DIR = os.path.join(RUN_DIR, 'plots')
os.makedirs(PLOT_DIR, exist_ok=True)

cmd = [
    sys.executable, 'scripts/visualize.py',
    '--log-json', log_path,
    '--out-dir', PLOT_DIR,
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for name in ['reward_curve.png', 'lambda_curve.png', 'phi_c_curve.png',
             'policy_loss.png', 'bc_loss_curve.png']:
    p = Path(PLOT_DIR) / name
    if p.exists():
        print(f'--- {name} ---')
        display(Image(filename=str(p)))
    else:
        print(f'(skipped {name} \u2014 not present in log)')

## 9. Placement heatmap (policy rollout)

Rolls the policy out greedily on one board and visualizes *where* the
policy likes to place components. Useful for sanity-checking that the
model has learned a sensible spatial prior.

In [ ]:
HEATMAP_BOARD = cfg_dict['env']['board_paths'][0]
HEATMAP_EPISODES = 8

cmd = [
    sys.executable, 'scripts/visualize.py',
    '--heatmap',
    '--checkpoint', FINAL_CKPT,
    '--config', CONFIG,
    '--board', HEATMAP_BOARD,
    '--heatmap-episodes', str(HEATMAP_EPISODES),
    '--out-dir', PLOT_DIR,
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)

heatmap_path = Path(PLOT_DIR) / 'placement_heatmap.png'
if heatmap_path.exists():
    display(Image(filename=str(heatmap_path)))

## 10. Final report

Composes a self-contained Markdown summary from the run directory. Use
the cell below to download the whole run as a single zip.

In [ ]:
from datetime import datetime

def _maybe(x, fmt='{:.4f}'):
    return fmt.format(x) if isinstance(x, (int, float)) else 'n/a'

n_iters = sum(1 for _ in open(log_path)) if os.path.exists(log_path) else 0
report = []
report.append('# SCPT-RL Run Report\n')
report.append(f'- **Generated**: {datetime.utcnow().isoformat(timespec="seconds")}Z')
report.append(f'- **Platform**: {PLATFORM}  |  **Device**: {DEVICE}')
report.append(f'- **Config**: `{CONFIG}`')
report.append(f'- **Run dir**: `{RUN_DIR}`')
report.append(f'- **Log lines**: {n_iters}')
report.append(f'- **Final checkpoint**: `{FINAL_CKPT}`\n')

report.append('## Evaluation\n')
report.append('| Metric | Value |')
report.append('|--------|------:|')
report.append(f'| Mean episode reward | {_maybe(eval_results.get("mean_reward"))} |')
report.append(f'| BC eval loss       | {_maybe(eval_results.get("bc_eval_loss"))} |')
report.append(f'| Boards evaluated   | {eval_results.get("n_boards", 0)} |')
report.append(f'| Episodes / board   | {N_EPISODES} |\n')

if eval_results.get('mean_costs'):
    report.append('### Mean costs (lower is better)\n')
    report.append('| Constraint | Mean |')
    report.append('|------------|-----:|')
    for k, v in eval_results['mean_costs'].items():
        report.append(f'| {k} | {_maybe(v)} |')
    report.append('')

if eval_results.get('checkpoint_lambdas'):
    report.append('### Lagrangian multipliers (frozen at checkpoint)\n')
    report.append('| Constraint | \u03bb |')
    report.append('|------------|------:|')
    for k, v in eval_results['checkpoint_lambdas'].items():
        report.append(f'| {k} | {_maybe(v, "{:.6f}")} |')
    report.append('')

if eval_results.get('per_board'):
    report.append('### Per-board breakdown\n')
    report.append('| Board | Mean reward | Mean costs |')
    report.append('|-------|------------:|-----------|')
    for row in eval_results['per_board']:
        board = os.path.basename(row.get('board', '?'))
        rw = _maybe(row.get('mean_reward'))
        costs = ', '.join(f'{k}={_maybe(v)}' for k, v in (row.get('mean_costs') or {}).items())
        report.append(f'| {board} | {rw} | {costs} |')
    report.append('')

report.append('## Plots\n')
for name in ['reward_curve.png', 'lambda_curve.png', 'phi_c_curve.png',
             'policy_loss.png', 'bc_loss_curve.png', 'placement_heatmap.png']:
    p = Path(PLOT_DIR) / name
    if p.exists():
        report.append(f'### {name}\n\n![]({p.as_posix()})\n')

report_md = '\n'.join(report)
report_path = Path(RUN_DIR) / 'REPORT.md'
report_path.write_text(report_md)
print(f'Wrote {report_path}  ({len(report_md)} chars)')

In [ ]:
from IPython.display import Markdown, display
display(Markdown(report_md))

## 11. Download the run

Bundles checkpoints, log, plots, and the Markdown report into a single zip
and exposes it through the browser's download dialog. On Kaggle the zip
is written to `/kaggle/working` and is also visible in the output panel.

In [ ]:
import shutil
ZIP_PATH = '/content/scpt-rl-run.zip' if IS_COLAB else '/kaggle/working/scpt-rl-run.zip' if IS_KAGGLE else 'scpt-rl-run.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', root_dir=os.getcwd(), base_dir=RUN_DIR)
print(f'Run zipped to {ZIP_PATH}  ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)')

if IS_COLAB:
    from google.colab import files  # type: ignore
    files.download(ZIP_PATH)
else:
    print('File is at', ZIP_PATH, '\u2014 download it from the file browser.')